# CARA: VGGT front-end experiment on Google Colab (ETH3D, no login needed)

Runs VGGT on image pairs from the ETH3D high-resolution multi-view benchmark under clean, fog, low-light and rain
conditions, and saves small `.npz` files. Send the final zip to Claude; the analysis runs elsewhere.

**Before you start:** Runtime > Change runtime type > **GPU**. Upload `CARA_GPU_kit.zip` to `MyDrive/CARA/`.
You need about 6 GB free in Google Drive (the ETH3D download is kept there so later sessions do not re-download it).

**Disconnects are safe:** outputs go to Drive and re-running a cell skips finished files. After a disconnect, run cells 1-4 again, then continue.

Design: 13 ETH3D scenes, split by scene: 6 for calibration, 7 for testing (`splits/eth3d_*.txt`). Fog is rendered with the
atmospheric scattering model using depth predicted by VGGT on the clean image (used only to draw the fog).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive and working folders
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/CARA'          # persistent (outputs, zips)
LOCAL = '/content/local'                       # fast, lost on disconnect (unzipped images)
os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
FREE_TIER = True    # True: reduced workload that fits free T4 sessions; False: full protocol (Colab Pro / L4 / A100)

In [ ]:
# 3. Install VGGT and the CARA kit (about 2 minutes)
# VGGT's packaging pins numpy<2, which downgrades Colab's numpy and breaks other packages.
# The part of VGGT we use (model, image loading, pose decoding) works with Colab's numpy 2,
# so we install VGGT WITHOUT its pinned dependencies and add only what it needs.
%cd /content
!test -d vggt || git clone -q https://github.com/facebookresearch/vggt.git
!pip -q install --no-deps -e /content/vggt
!pip -q install einops safetensors huggingface_hub pycolmap==4.2.0 gtsam
!unzip -oq {WORK}/CARA_GPU_kit.zip -d /content/cara_sfm
%cd /content/cara_sfm/gpu_kit
!python -c "import numpy, torch, vggt; from vggt.models.vggt import VGGT; print('numpy', numpy.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| VGGT import OK')"


In [ ]:
# 4. ETH3D data (5.5 GB, direct download, kept on Drive; extracted to fast local disk each session)
import os
z = f'{WORK}/multi_view_training_dslr_undistorted.7z'
if not os.path.exists(z):
    !wget -c -q --show-progress -O {z}.part https://www.eth3d.net/data/multi_view_training_dslr_undistorted.7z && mv {z}.part {z}
!apt-get -qq install -y p7zip-full > /dev/null
if not os.path.exists(f'{LOCAL}/eth3d/facade'):
    !mkdir -p {LOCAL}/eth3d && 7z x -y -o{LOCAL}/eth3d {z} > /dev/null
!ls {LOCAL}/eth3d
!find {LOCAL}/eth3d/facade -maxdepth 2 | head -8

In [ ]:
# 5. Settings (pre-registered; do not change after seeing results)
ETH = f'{LOCAL}/eth3d'
OUT = f'{WORK}/out_eth3d'
CAL = dict(n_sub=12, per_scene=24, k=6, batch=8)       # 6 scenes x 24 = 144 calibration sub-scenes (>= 140 needed)
TEST = dict(n_sub=12, per_scene=10, k=6, batch=8, conds='clean,fog0.7,fog0.5,fog0.3,low0.3,low0.15,rain0.5')
# If a run fails with 'CUDA out of memory', set batch=4 in both lines and re-run (finished files are kept).

In [ ]:
# 6. Timing test (one sub-scene). The first run downloads the VGGT-1B weights (~5 GB); run it twice, use the 2nd ETA.
!python frontend_vggt.py --dataset eth3d --root {ETH} --out {WORK}/timing --role calib \
   --scenes-file splits/eth3d_calib.txt --n-sub {CAL['n_sub']} --per-scene 1 --k {CAL['k']} --batch {CAL['batch']} --conds clean 2>&1 | tail -3

In [ ]:
# 7. Stage 1a: calibration sub-scenes (clean only). Resumable.
!python frontend_vggt.py --dataset eth3d --root {ETH} --out {OUT} --role calib \
   --scenes-file splits/eth3d_calib.txt --n-sub {CAL['n_sub']} --per-scene {CAL['per_scene']} --k {CAL['k']} --batch {CAL['batch']} --conds clean

In [ ]:
# 8. Sanity check. STOP and send the output to Claude if the clean median error is above ~15 deg,
#    if the 'transposed' error is smaller than the normal one, or if the Spearman value is positive.
!python check_frontend.py --in {OUT}

In [ ]:
# 9. Stage 1b: test sub-scenes (7 other scenes, 7 conditions). Resumable.
!python frontend_vggt.py --dataset eth3d --root {ETH} --out {OUT} --role test --seed 1 \
   --scenes-file splits/eth3d_test.txt --n-sub {TEST['n_sub']} --per-scene {TEST['per_scene']} --k {TEST['k']} --batch {TEST['batch']} --conds {TEST['conds']}

In [ ]:
# 10. Look at a few degraded images (fog / low light / rain should look plausible)
import glob, random
from PIL import Image
from IPython.display import display
fs = sorted(glob.glob(f'{OUT}/_degraded/[0-9a-f]*.jpg')); random.seed(0)
for f in random.sample(fs, min(4, len(fs))):
    im = Image.open(f); im.thumbnail((480, 480)); display(im)

In [ ]:
# 11. Package results for Claude (a few MB): npz files, manifests, sanity check, GPU name, 12 example images
!python check_frontend.py --in {OUT} > {WORK}/check_frontend.txt; cat {WORK}/check_frontend.txt
!nvidia-smi --query-gpu=name --format=csv > {WORK}/gpu.txt
!cd {WORK} && rm -f CARA_frontend_results.zip && zip -qr CARA_frontend_results.zip out_eth3d check_frontend.txt gpu.txt -x 'out_eth3d/_degraded/*'
!cd {WORK} && ls out_eth3d/_degraded | grep -v '^clean_' | head -12 | sed 's#^#out_eth3d/_degraded/#' | zip -q CARA_frontend_results.zip -@
!ls -la {WORK}/CARA_frontend_results.zip
print('Send MyDrive/CARA/CARA_frontend_results.zip to Claude.')